In [13]:
import sys
import os
import numpy as np
import pandas as pd
import torch
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset
from sklearn.model_selection import train_test_split, KFold

In [14]:
sys.path.append('../../')

from modules.cross_attentionb import CrossAttentionB
from modules.dataloader import load_npy_files
from modules.classifier import DenseLayer, BCELoss, CustomLoss, BCEWithLogits
from modules.linear_transformation import LinearTransformations
from modules.output_max import output_max
from evaluation_validation.train_model import train_model
from evaluation_validation.evaluate_model import evaluate_model
from evaluation_validation.test_model import test_model

In [15]:
# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### Data Loading

In [16]:
class MultimodalDataset(Dataset):
    def __init__(self, id_label_df, text_features, audio_features, video_features):
        self.id_label_df = id_label_df
        
        # Convert feature lists to dictionaries for fast lookup
        self.text_features = {os.path.basename(file).split('.')[0]: tensor for file, tensor in text_features}
        self.audio_features = {os.path.basename(file).split('_')[1].split('.')[0]: tensor for file, tensor in audio_features}
        self.video_features = {os.path.basename(file).split('_')[0]: tensor for file, tensor in video_features}

        # List to store missing files
        self.missing_files = []

        # Filter out entries with missing files
        self.valid_files = self._filter_valid_files()


    def _filter_valid_files(self):
        valid_files = []
        for idx in range(len(self.id_label_df)):
            imdbid = self.id_label_df.iloc[idx]['IMDBid']

            # Check if the IMDBid exists in each modality's features
            if imdbid in self.text_features and imdbid in self.audio_features and imdbid in self.video_features:
                valid_files.append(idx)
            else:
                self.missing_files.append({'IMDBid': imdbid})

        # # Print missing files after checking all
        # if self.missing_files:
        #     print("Missing files:")
        #     for item in self.missing_files:
        #         print(f"IMDBid: {item['IMDBid']}")
        #     print(f"Total IMDB IDs with missing files: {len(self.missing_files)}")
        # else:
        #     print("No missing files.")

        return valid_files

    def __len__(self):
        return len(self.valid_files)

    def __getitem__(self, idx):
        # Get the original index from the filtered valid files
        original_idx = self.valid_files[idx]
        imdbid = self.id_label_df.iloc[original_idx]['IMDBid']
        label = self.id_label_df.iloc[original_idx]['Label']

        # Retrieve data from the loaded features
        text_data = self.text_features.get(imdbid, torch.zeros((1024,)))
        audio_data = self.audio_features.get(imdbid, torch.zeros((1, 197, 768)))
        video_data = self.video_features.get(imdbid, torch.zeros((95, 768)))
        
        # Define label mapping
        label_map = {'red': 0, 'green': 1} 
        
        # Convert labels to tensor using label_map
        try:
            label_data = torch.tensor([label_map[label]], dtype=torch.float32)  # Ensure labels are integers
        except KeyError as e:
            print(f"Error: Label '{e}' not found in label_map.")
            raise
        
        # Debugging output
        if label_data.shape[0] == 0:
            print(f"Empty target for IMDBid {imdbid} at index {idx}")

        return text_data, audio_data, video_data, label_data


In [17]:
def collate_fn(batch):
    text_data, audio_data, video_data, label_data = zip(*batch)

    # Convert lists to tensors
    text_data = torch.stack(text_data)
    audio_data = torch.stack(audio_data)

    # Padding for video data
    # Determine maximum length of video sequences in the batch
    video_lengths = [v.size(0) for v in video_data]
    max_length = max(video_lengths)

    # Pad video sequences to the maximum length
    video_data_padded = torch.stack([
        F.pad(v, (0, 0, 0, max_length - v.size(0)), "constant", 0)
        for v in video_data
    ])

    # Convert labels to tensor and ensure the shape [batch_size, 1]
    label_data = torch.stack(label_data)  # Convert list of tensors to a single tensor

    return text_data, audio_data, video_data_padded, label_data

In [18]:
# Load the labels DataFrame
id_label_df = pd.read_excel('../../misc/MM-Trailer_dataset.xlsx')

# Define the directories
text_features_dir = '../../misc/text_features'
audio_features_dir = '../../misc/audio_features'
video_features_dir = '../../misc/video_features'

# Load the feature vectors from each directory
text_features = load_npy_files(text_features_dir)
audio_features = load_npy_files(audio_features_dir)
video_features = load_npy_files(video_features_dir)

print(f"Number of text feature vectors loaded: {len(text_features)}")
print(f"Number of audio feature vectors loaded: {len(audio_features)}")
print(f"Number of video feature vectors loaded: {len(video_features)}")

# Drop unnecessary columns
id_label_df = id_label_df.drop(columns=['Movie Title', 'URL'])

# Splitting data for training, validation, and testing
train_df, val_test_df = train_test_split(id_label_df, test_size=0.3, random_state=42)

# Further splitting remaining set into validation and test sets
val_df, test_df = train_test_split(val_test_df, test_size=0.5, random_state=42)

print("-" * 30)

print("training, validation, and testing shapes: ")
print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

# Create datasets
train_dataset = MultimodalDataset(train_df, text_features, audio_features, video_features)
val_dataset = MultimodalDataset(val_df, text_features, audio_features, video_features)
test_dataset = MultimodalDataset(test_df, text_features, audio_features, video_features)

# Create DataLoaders
train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=0, collate_fn=collate_fn)
val_dataloader = DataLoader(val_dataset, batch_size=16, shuffle=True, num_workers=0, collate_fn=collate_fn)
test_dataloader = DataLoader(test_dataset, batch_size=16, shuffle=True, num_workers=0, collate_fn=collate_fn)

# Combine all data for K-fold cross-validation
full_dataset = MultimodalDataset(id_label_df, text_features, audio_features, video_features)

Number of text feature vectors loaded: 1353
Number of audio feature vectors loaded: 1353
Number of video feature vectors loaded: 1353
------------------------------
training, validation, and testing shapes: 
(1002, 2)
(215, 2)
(215, 2)


### SMCA Model Classes

In [20]:
# Stage 1 of SMCA
def SMCAStage1(modalityAlpha, modalityBeta, d_out_kq, d_out_v, device):
    cross_attn = CrossAttentionB(modalityAlpha.shape[-1], modalityBeta.shape[-1], d_out_kq, d_out_v).to(device)
    modalityAlphaBeta = cross_attn(modalityAlpha, modalityBeta)
    return modalityAlphaBeta

In [21]:
# Stage 2 of SMCA - Model A: Stage 1 Output as Query
def SMCAStage2_ModelA(modalityAlphaBeta, modalityGamma, d_out_kq, d_out_v, device):
    cross_attn = CrossAttentionB(modalityAlphaBeta.shape[-1], modalityGamma.shape[-1], d_out_kq, d_out_v).to(device)
    multimodal_representation = cross_attn(modalityAlphaBeta, modalityGamma)
    return multimodal_representation

In [22]:
class SMCAModelA(nn.Module):
    def __init__(self, d_out_kq, d_out_v, device):
        super(SMCAModelA, self).__init__()
        self.d_out_kq = d_out_kq
        self.d_out_v = d_out_v
        self.device = device
    
    def forward(self, modalityAlpha, modalityBeta, modalityGamma, device):
        # Stage 1: Cross attention between modalityAlpha and modalityBeta
        modalityAlphaBeta = SMCAStage1(modalityAlpha, modalityBeta, self.d_out_kq, self.d_out_v, device)

        # Stage 2: Cross attention with modalityAlphaBeta (as query) and modalityGamma (as key-value)
        multimodal_representation = SMCAStage2_ModelA(modalityAlphaBeta, modalityGamma, self.d_out_kq, self.d_out_v, device)
 
        # Flatten the output
        return torch.flatten(multimodal_representation, start_dim=1)  # Flatten all dimensions except batch
        

### Model Training Functions

In [23]:
def get_optimizer(parameters, lr=1e-3):
    # Create an optimizer, for example, Adam
    return optim.Adam(parameters, lr=lr)

### Fusion Model A

In [24]:
if __name__ == "__main__":
    torch.manual_seed(42)

    # Device configuration
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    # Determine the output dimensions
    output_dim = 768

    # Initialize the SMCA model A
    model = SMCAModelA(512, 256, device) # Dimension for d_out_kq and d_out_v
    model.to(device)  # Move the model to the correct device

    # Loop through the dataloaders to determine the largest output size
    max_output_size = output_max(model, train_dataloader, device)

    # Initialize the DenseLayer with the largest output size
    dense_layer = DenseLayer(input_size=max_output_size).to(device)  # Initialize and move to the correct device

    # Define the loss function and optimizer
    criterion = BCEWithLogits()  # Use appropriate loss function
    
    for param in model.parameters():
        if param.grad is None:
            print("No gradient for:", param)
    optimizer = get_optimizer(list(model.parameters()) + list(dense_layer.parameters()))


    # Training loop
    num_epochs = 30  # Set the number of epochs you want to train for
   
    for epoch in range(num_epochs):
        print("-" * 30)
        print(f"Epoch {epoch + 1}/{num_epochs}")

        # Ensure you have a dataloader that yields inputs and targets
        train_loss = train_model(model=model, dense_layer=dense_layer, dataloader=train_dataloader, criterion=criterion, optimizer=optimizer, device=device)

        # Validate step
        val_loss, precision, recall, f1_score = evaluate_model(model=model, dense_layer=dense_layer, dataloader=val_dataloader, criterion=criterion, device=device)

        print(f"Training Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}")

    # Testing the model
    print("-" * 30)
    print("Testing the model on the test set...")
    test_loss, test_precision, test_recall, test_f1_score = test_model(model=model, dense_layer=dense_layer, dataloader=test_dataloader, criterion=criterion, device=device)


Device: cpu
------------------------------
Epoch 1/30
Precision: 0.8125
Recall: 0.2690
F1 Score: 0.4041
Training Loss: 0.7051, Validation Loss: 0.7529
------------------------------
Epoch 2/30
Precision: 0.8438
Recall: 0.1862
F1 Score: 0.3051
Training Loss: 0.7717, Validation Loss: 0.7683
------------------------------
Epoch 3/30
Precision: 0.7031
Recall: 0.3103
F1 Score: 0.4306
Training Loss: 0.7823, Validation Loss: 0.8284
------------------------------
Epoch 4/30
Precision: 0.7625
Recall: 0.4207
F1 Score: 0.5422
Training Loss: 0.7998, Validation Loss: 0.8975
------------------------------
Epoch 5/30
Precision: 0.7391
Recall: 0.3517
F1 Score: 0.4766
Training Loss: 0.8008, Validation Loss: 0.9234
------------------------------
Epoch 6/30
Precision: 0.7228
Recall: 0.5034
F1 Score: 0.5935
Training Loss: 0.7304, Validation Loss: 0.7636
------------------------------
Epoch 7/30
Precision: 0.7294
Recall: 0.4276
F1 Score: 0.5391
Training Loss: 0.7961, Validation Loss: 0.8457
---------------